In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
os.chdir("..")

In [2]:
from src.data.download import load_config, extract_corpus

cfg = load_config("configs/base.yaml")
train = [e for e in cfg["data"]["corpora"] if e["split"] == "train"]
[e["name"] for e in train]

['news_commentary', 'europarl', 'commoncrawl']

In [3]:
entry = train[0]                      # news_commentary
src_file, tgt_file = extract_corpus(entry, cfg)

src_lens, tgt_lens, ratios = [], [], []

with open(src_file, encoding="utf-8", newline="\n") as fs, \
     open(tgt_file, encoding="utf-8", newline="\n") as ft:
    for s, t in zip(fs, ft):
        a, b = len(s.split()), len(t.split())
        src_lens.append(a)
        tgt_lens.append(b)
        if a and b:
            ratios.append(max(a, b) / min(a, b))

In [4]:
def pct(vals, label):
    vals = sorted(vals)
    n = len(vals)
    pts = [vals[int(p * n)] for p in (0.01, 0.5, 0.95, 0.99, 0.999)]
    print(label, "1%/50%/95%/99%/99.9%:", pts, "max:", vals[-1])

pct(src_lens, "en")
pct(tgt_lens, "de")
pct(ratios, "ratio")

en 1%/50%/95%/99%/99.9%: [3, 20, 42, 54, 76] max: 171
de 1%/50%/95%/99%/99.9%: [3, 21, 44, 56, 78] max: 193
ratio 1%/50%/95%/99%/99.9%: [1.0, 1.1081081081081081, 1.4285714285714286, 1.76, 2.909090909090909] max: 74.0


In [5]:
for entry in train:
    src_file, tgt_file = extract_corpus(entry, cfg)
    src_lens, tgt_lens, ratios = [], [], []
    with open(src_file, encoding="utf-8", newline="\n") as fs, \
         open(tgt_file, encoding="utf-8", newline="\n") as ft:
        for s, t in zip(fs, ft):
            a, b = len(s.split()), len(t.split())
            src_lens.append(a)
            tgt_lens.append(b)
            if a and b:
                ratios.append(max(a, b) / min(a, b))
    print("===", entry["name"], len(src_lens))
    pct(src_lens, "en")
    pct(tgt_lens, "de")
    pct(ratios, "ratio")

=== news_commentary 201288
en 1%/50%/95%/99%/99.9%: [3, 20, 42, 54, 76] max: 171
de 1%/50%/95%/99%/99.9%: [3, 21, 44, 56, 78] max: 193
ratio 1%/50%/95%/99%/99.9%: [1.0, 1.1081081081081081, 1.4285714285714286, 1.76, 2.909090909090909] max: 74.0
=== europarl 1920209
en 1%/50%/95%/99%/99.9%: [1, 22, 53, 73, 107] max: 668
de 1%/50%/95%/99%/99.9%: [2, 21, 49, 68, 100] max: 426
ratio 1%/50%/95%/99%/99.9%: [1.0, 1.125, 1.5, 2.0, 4.0] max: 71.0
=== commoncrawl 2399123
en 1%/50%/95%/99%/99.9%: [6, 18, 45, 64, 103] max: 4225
de 1%/50%/95%/99%/99.9%: [6, 17, 41, 58, 94] max: 2937
ratio 1%/50%/95%/99%/99.9%: [1.0, 1.2, 2.5555555555555554, 4.333333333333333, 8.181818181818182] max: 175.0909090909091


In [6]:
entry = train[2]                      # commoncrawl
src_file, tgt_file = extract_corpus(entry, cfg)

pairs = []
with open(src_file, encoding="utf-8", newline="\n") as fs, \
     open(tgt_file, encoding="utf-8", newline="\n") as ft:
    for s, t in zip(fs, ft):
        a, b = len(s.split()), len(t.split())
        if a and b and max(a, b) / min(a, b) > 2.5:
            pairs.append((a, b, s.strip()[:80], t.strip()[:80]))

print(len(pairs))
for p in pairs[:15]:
    print(p)

124030
(17, 6, 'Get VTeacher, a screensaver that displays words and phrases you are trying to le', 'MP3 player plays multi format music.')
(16, 44, 'The English-German Pro Dictionary contains over 50,813 words and 23,343 articles', 'Take control of your life, and easily organize your tasks, meetings, keep track ')
(51, 14, 'For example, you’ve created a layer in Photoshop to give your image an antiqued ', 'Hier werden sofort in einer Miniaturansicht die einzelnen Ebenen zusammen mit de')
(12, 35, 'AKVIS Chameleon is a fun to use tool for photo collage creation.', 'AKVIS Chameleon ist ein wunderbarer Plugin für Erstellung von Fotocollagen mit a')
(12, 40, 'AKVIS Coloriage allows colorizing B&W photos and replacing colors in color photo', 'Das Programm ändert die Farben eines Fotos: von der Einfärbung S/ W-Fotos bis hi')
(9, 37, 'AKVIS Magnifier allows resizing images without loss in quality.', 'AKVIS Magnifier erlaubt es, Fotos zu vergrößern, ohne dass das Bild an Schärfe v')
(23, 8, "T

In [11]:
import py3langid as langid
print(langid.classify("This is an English sentence."))
print(langid.classify("Das ist ein deutscher Satz."))
print(langid.classify("OK."))

('en', np.float32(-77.17697))
('de', np.float32(-136.60867))
('en', np.float32(9.06184))


In [13]:
from src.data.clean import is_expected_language

entry = train[2]
src_file, tgt_file = extract_corpus(entry, cfg)

with open(src_file, encoding="utf-8", newline="\n") as fs, \
     open(tgt_file, encoding="utf-8", newline="\n") as ft:
    for i, (s, t) in enumerate(zip(fs, ft), start=1):
        if i > 15:
            break
        print(i, is_expected_language(s.strip(), t.strip(), "en", "de"))

'des Goldes  hinwiesen' -> 'des Goldes hinwiesen'
'verzehnfachen?  ' -> 'verzehnfachen?'
'it\xadwill' -> 'itwill'
'„Freak Peak“' -> '"Freak Peak"'
'gold’s risks' -> "gold's risks"
'10.000 Dollar' -> '10.000 Dollar'
'$10,000' -> '$10,000'
1 True
2 True
3 False
4 True
5 True
6 True
7 True
8 False
9 False
10 False
11 False
12 False
13 False
14 False
15 False


In [14]:
print(langid.classify("a fire restant repair cement for fire places, ovens, open fireplaces etc."))
print(langid.classify("feuerfester Reparaturkitt für Feuerungsanlagen, Öfen, offene Feuerstellen etc."))

('fr', np.float32(-76.73909))
('de', np.float32(-261.92242))


In [15]:
from src.data.clean import is_expected_language

entry = train[0]                      # news_commentary — the clean corpus
src_file, tgt_file = extract_corpus(entry, cfg)

rejected = []
n = 0
with open(src_file, encoding="utf-8", newline="\n") as fs, \
     open(tgt_file, encoding="utf-8", newline="\n") as ft:
    for s, t in zip(fs, ft):
        if n >= 500:
            break
        s, t = s.strip(), t.strip()
        if not (s and t):
            continue
        n += 1
        if not is_expected_language(s, t, "en", "de"):
            rejected.append((s[:70], t[:70]))

print(f"{len(rejected)} / {n} rejected = {100*len(rejected)/n:.1f}%")
for r in rejected[:15]:
    print(r)

2 / 500 rejected = 0.4%
('\xadA New European Growth Agenda', 'Eine neue europäische Wachstumsagenda')
('“Neo-Ottoman” Turkey?', 'Eine „neuottomanische“ Türkei?')
